# Case 2 — NACA 4412 Airfoil: POD-AS-PRS Workflow

This notebook reproduces the lift-coefficient ($C_l$) surrogate study for a
NACA 4412 airfoil at $Re = 1.75\times10^4$ using the full POD-AS-PRS pipeline:

1. **Preprocessing** – merge Nek5000 snapshots and compute vorticity
2. **POD** – decompose vorticity snapshots via SVD (with spatial-region filter)
3. **ResNet** – train a fully-connected ResNet to map POD coefficients → $C_l$
4. **Gradient analysis** – compute autograd gradients and validate against FD
5. **Active Subspaces (AS)** – identify the dominant input directions
6. **Polynomial Response Surface (PRS)** – fit and evaluate the low-dimensional surrogate


## 0 · Setup

In [1]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch

from core.pod_engine       import POD_SVD
from core.resnet_model     import ResNet
from core.resnet_trainer   import (set_random_seed, load_or_train,
                                    evaluate_and_save_metrics,
                                    plot_loss_curve, plot_prediction_comparison,
                                    compute_all_gradients)
from core.gradient_analysis import compare_gradients_nature_style_dataset
from utils.data_loader      import (load_and_preprocess_data, denormalise,
                                     load_pod_vis_data)
from utils.visualization    import (plot_pod_importance, plot_response_surface_2d,
                                     validate_response_surface,
                                     compare_rom_fom_predictions,
                                     plot_polynomial_cv,
                                     plot_subspace_polynomial_heatmap,
                                     plot_interaction_heatmap,
                                     plot_pod_energy,
                                     plot_eigenvalues,
                                     plot_pod_modes_and_coeffs,
                                     plot_pod_phase_space_triangle,
                                     plot_mesh_and_vorticity,
                                     plot_qoi)
import lib.active_subspaces as ac

set_random_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Random seed set to: 42
Device: cuda


## 1 · Data Paths
Adjust these paths to match your local directory layout.

In [2]:
FLOW_DATA_PATH = '../data/Case2_NACA4412/flow_field_data.npz'
QOI_DATA_PATH  = '../data/Case2_NACA4412/lift_coefficient_600-800_truncated.dat'
RESULTS_DIR    = '../results/Case2_NACA4412'
MODEL_PATH     = os.path.join(RESULTS_DIR, 'resnet_model.pth')

NUM_POD_COEFFS = 500     # NACA 4412: match legacy train.py num_pod_coeffs=500
NUM_AS_MODES   = 150     # AS analysis uses first 150 modes (legacy main.py line 23)
os.makedirs(RESULTS_DIR, exist_ok=True)

## 1.5 · Computational Mesh and Vorticity Field

In [3]:
NEK_FILE = '/mnt/data/bak/HD5/NekExamples/NACA4412/airfoil0.f00001'

plot_mesh_and_vorticity(
    flow_data_path=FLOW_DATA_PATH,
    geometry='naca4412',
    nek_data_path=NEK_FILE,
    snapshot_idx=100,
    save_dir=os.path.join(RESULTS_DIR, 'Mesh'),
)

Mesh/vorticity plot saved to ../results/Case2_NACA4412/Mesh/mesh.[jpg|pdf]


<!-- TODO: Delete this cell — QoI plot is now in examples/plot_qoi.py -->

## 2 · Load Data and Run POD 

In [ ]:
(
    train_loader, val_loader, test_loader,
    pod_coeffs, pod_coeffs_norm,
    pod_mean, pod_std,
    qoi_mean, qoi_std,
    pod_min, pod_max,
    qoi_min, qoi_max,
) = load_and_preprocess_data(
    flow_data_path=FLOW_DATA_PATH,
    qoi_data_path=QOI_DATA_PATH,
    num_pod_coeffs=NUM_POD_COEFFS,
    train_ratio=0.8,
    val_ratio=0.1,
    batch_size=32,
    apply_region_filter=True,           
    pod_save_dir=os.path.join(RESULTS_DIR, 'POD'),
)

print(f'POD coefficients shape : {pod_coeffs.shape}')
print(f'Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

# POD visualisation — energy spectrum, spatial modes + time coefficients, phase portrait
pod_vis = load_pod_vis_data(
    pod_save_dir=os.path.join(RESULTS_DIR, 'POD'),
    flow_data_path=FLOW_DATA_PATH,
    apply_region_filter=True,
)

plot_pod_energy(
    pod_vis['Ds'], num_modes=pod_vis['An'].shape[0] - 1,
    save_dir=os.path.join(RESULTS_DIR, 'POD'),
    geometry='naca4412',
)

plot_eigenvalues(
    pod_vis['S'], num_values=pod_vis['An'].shape[0] - 1,
    save_dir=os.path.join(RESULTS_DIR, 'POD'),
    geometry='naca4412',
)

plot_pod_modes_and_coeffs(
    pod_vis['PhiU'], pod_vis['An'],
    pod_vis['original_shape'],
    pod_vis['x_grid'], pod_vis['y_grid'],
    num_modes=10, geometry='naca4412',
    region_mask=pod_vis['region_mask'],
    save_dir=os.path.join(RESULTS_DIR, 'POD'),
)

plot_pod_phase_space_triangle(
    pod_vis['An'], num_modes=10,
    save_dir=os.path.join(RESULTS_DIR, 'POD'),
    geometry='naca4412',
)


Loading flow field data...
Flow data keys: ['coords', 'times', 'velocity', 'pressure', 'vorticity', 'vorticity_grid_x', 'vorticity_grid_y', 'n_timesteps', 'n_elements', 'n_components', 'n_points', 'start_time', 'end_time', 'start_idx', 'end_idx', 'mesh_limits_x', 'mesh_limits_y', 'vorticity_nx', 'vorticity_ny']


Vorticity array shape: (5000, 399, 698)
Grid range: X=[-2.00, 6.75], Y=[-2.50, 2.50]
Spatial filter: 278502 / 278502 points retained (100.00%)
Filtered vorticity shape: (5000, 278502)
Loading cached POD data from: ../results/Case2_NACA4412/POD/pod_data.npz

Loading QoI data from: ../data/Case2_NACA4412/lift_coefficient_600-800_truncated.dat
QoI shape: (5000, 1), range: [0.521536, 1.077456]
POD coefficients used: (5000, 500)
QoI values used:       (5000, 1)
POD coefficients shape : (5000, 500)
Train batches: 125, Val: 16, Test: 16
POD energy plot saved to ../results/Case2_NACA4412/POD/pod_energy.[jpg|pdf]
POD eigenvalue plot saved to ../results/Case2_NACA4412/POD/pod_eigenvalues_log.[jpg|pdf]
POD modes & coefficients plot saved to ../results/Case2_NACA4412/POD/pod_modes_and_coeffs.[jpg|pdf]
POD phase-space triangle plot saved to ../results/Case2_NACA4412/POD/pod_phase_space_triangle.[jpg|pdf]


## 3 · Build and Train the ResNet Surrogate

In [5]:
set_random_seed(42)  

# Match legacy architecture: hidden_size=128, num_blocks=7, dropout_rate=0.3
NUM_BLOCKS = 7
model = ResNet(input_size=NUM_POD_COEFFS, hidden_size=128,
               num_blocks=NUM_BLOCKS, dropout_rate=0.3).to(DEVICE)
print(model)

model, train_losses, val_losses = load_or_train(
    model, train_loader, val_loader, DEVICE,
    model_save_path=MODEL_PATH,
    num_epochs=1000,
    patience=100,
    lr=0.001,
    weight_decay=1e-4,   
)

plot_loss_curve(train_losses, val_losses, patience=100,
                results_dir=RESULTS_DIR)

Random seed set to: 42
ResNet(
  (input_layer): Sequential(
    (0): Linear(in_features=500, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (res_blocks): ModuleList(
    (0-6): 7 x ResidualBlock(
      (block): Sequential(
        (0): Linear(in_features=128, out_features=128, bias=True)
        (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Dropout(p=0.1, inplace=False)
        (4): Linear(in_features=128, out_features=128, bias=True)
        (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (relu): ReLU()
    )
  )
  (output_layer): Sequential(
    (0): Dropout(p=0.3, inplace=False)
    (1): Linear(in_features=128, out_features=64, bias=True)
    (2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ReLU()
    (4): Dropout(p=0.3, in

## 4 · Evaluate the Surrogate

In [6]:
denorm = lambda v: denormalise(v, qoi_mean, qoi_std)

metrics = evaluate_and_save_metrics(
    model,
    loaders=[train_loader, val_loader, test_loader],
    split_names=['Train', 'Validation', 'Test'],
    device=DEVICE,
    denorm_fn=denorm,
    results_dir=RESULTS_DIR,
)

Train: MSE=6.831757e-05, MAE=6.794237e-03, R²=0.995982, MaxRelErr=0.046546
Validation: MSE=6.361391e-04, MAE=2.042101e-02, R²=0.965398, MaxRelErr=0.117046
Test: MSE=1.348683e-03, MAE=2.785009e-02, R²=0.917464, MaxRelErr=0.194825
Metrics saved to ../results/Case2_NACA4412/metrics.txt


## 5 · Gradient Analysis (Autograd vs Finite Difference)

In [7]:
# Collect only training-set samples 
# (iterates train_loader only, not all 1000 samples)
pod_train_list = []
for inputs, _ in train_loader:
    pod_train_list.append(inputs)
pod_norm_torch = torch.cat(pod_train_list, dim=0)   # shape: (800, NUM_POD_COEFFS)

avg_err, max_err, timing = compare_gradients_nature_style_dataset(
    model, pod_norm_torch, device=DEVICE,
    h=1e-2, max_modes=20,
    save_path=os.path.join(RESULTS_DIR, 'gradient_comparison.pdf'),
    batch_size=16,   
)
print(f'Mean rel. error: {avg_err:.6f}, Max rel. error: {max_err:.6f}')
print(f'AD/FD speedup: {timing["speedup_ratio"]:.1f}x')


Dataset-level gradient comparison for 4000 samples...
  Batch 1/250 (samples 0–15)


  Batch 2/250 (samples 16–31)
  Batch 3/250 (samples 32–47)
  Batch 4/250 (samples 48–63)
  Batch 5/250 (samples 64–79)
  Batch 6/250 (samples 80–95)
  Batch 7/250 (samples 96–111)
  Batch 8/250 (samples 112–127)
  Batch 9/250 (samples 128–143)
  Batch 10/250 (samples 144–159)
  Batch 11/250 (samples 160–175)
  Batch 12/250 (samples 176–191)
  Batch 13/250 (samples 192–207)
  Batch 14/250 (samples 208–223)
  Batch 15/250 (samples 224–239)
  Batch 16/250 (samples 240–255)
  Batch 17/250 (samples 256–271)
  Batch 18/250 (samples 272–287)
  Batch 19/250 (samples 288–303)
  Batch 20/250 (samples 304–319)
  Batch 21/250 (samples 320–335)
  Batch 22/250 (samples 336–351)
  Batch 23/250 (samples 352–367)
  Batch 24/250 (samples 368–383)
  Batch 25/250 (samples 384–399)
  Batch 26/250 (samples 400–415)
  Batch 27/250 (samples 416–431)
  Batch 28/250 (samples 432–447)
  Batch 29/250 (samples 448–463)
  Batch 30/250 (samples 464–479)
  Batch 31/250 (samples 480–495)
  Batch 32/250 (samples 496–5

/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/core/gradient_analysis.py:327: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(x="POD Mode", y="Gradient Difference (AD - FD)",


Mean rel. error: 0.026183, Max rel. error: 100.000000
AD/FD speedup: 271.7x


## 6 · Compute Gradients for All Samples

In [8]:
gradients = compute_all_gradients(model, pod_coeffs, DEVICE, batch_size=32)
print(f'Gradient matrix shape: {gradients.shape}')

grad_save = os.path.join(RESULTS_DIR, 'pod_gradients.npy')
np.save(grad_save, gradients)
print(f'Saved to {grad_save}')

Normalised POD range: [-1.0000, 1.0000]
  Gradient batch 1/157 (samples 0–31)
  Gradient batch 2/157 (samples 32–63)
  Gradient batch 3/157 (samples 64–95)
  Gradient batch 4/157 (samples 96–127)
  Gradient batch 5/157 (samples 128–159)
  Gradient batch 6/157 (samples 160–191)
  Gradient batch 7/157 (samples 192–223)
  Gradient batch 8/157 (samples 224–255)
  Gradient batch 9/157 (samples 256–287)
  Gradient batch 10/157 (samples 288–319)
  Gradient batch 11/157 (samples 320–351)
  Gradient batch 12/157 (samples 352–383)
  Gradient batch 13/157 (samples 384–415)
  Gradient batch 14/157 (samples 416–447)
  Gradient batch 15/157 (samples 448–479)
  Gradient batch 16/157 (samples 480–511)
  Gradient batch 17/157 (samples 512–543)
  Gradient batch 18/157 (samples 544–575)
  Gradient batch 19/157 (samples 576–607)
  Gradient batch 20/157 (samples 608–639)
  Gradient batch 21/157 (samples 640–671)
  Gradient batch 22/157 (samples 672–703)
  Gradient batch 23/157 (samples 704–735)
  Gradient 

## 7 · Active Subspace Analysis

In [9]:
# TODO: Delete this cell — superseded by the corrected AS analysis cell below (uses NUM_AS_MODES=150 slicing)

In [10]:
# AS analysis uses first NUM_AS_MODES=150 modes (legacy main.py lines 26-50)
XX_as_min = np.min(pod_coeffs[:, :NUM_AS_MODES], axis=0)
XX_as_max = np.max(pod_coeffs[:, :NUM_AS_MODES], axis=0)
scale = (XX_as_max - XX_as_min) / 2.0
scale[scale < 1e-10] = 1.0
gradients_scaled = gradients[:, :NUM_AS_MODES] * scale

# Bootstrap-based AS computation — nboot=1000 
ss = ac.subspaces.Subspaces()
ss.compute(df=gradients_scaled, nboot=1000)

opts = ac.utils.plotters.plot_opts(savefigs=True)

# First 20 eigenvalues with sparse ticks (150 modes → many labels)
ac.utils.plotters.eigenvalues(
    ss.eigenvals[:20],
    e_br=ss.e_br[:20, :],
    out_label='$C_l$',
    opts=opts,
    figsize=(10, 8),
    sparse_xticks=True,
    save_path=os.path.join(RESULTS_DIR, 'eigenvalues.jpg'),
)
ac.utils.plotters.subspace_errors(
    ss.sub_br[:20, :], out_label='$C_l$', opts=opts
)

# Eigenvectors heatmap: first 20 modes × first 20 AS directions
ac.utils.plotters.eigenvectors_heatmap(
    ss.eigenvecs[:20, :20],
    out_label='$C_l$',
    opts=opts,
    save_path=os.path.join(RESULTS_DIR, 'eigenvectors_heatmap.jpg'),
)

# n_active=22 matches legacy main.py line 76
n_active = 22
ss.partition(n_active)
print(f'Active subspace dimension: {n_active}')
print(f'W1 (active directions) shape: {ss.W1.shape}')

Active subspace dimension: 22
W1 (active directions) shape: (150, 22)


## 8 · POD Mode Importance

In [11]:
# Load lift coefficient (raw physical values)
qoi_raw = np.loadtxt(QOI_DATA_PATH)
qoi_full = qoi_raw[:pod_coeffs.shape[0], 1]

# Project onto active subspace — use first NUM_AS_MODES (matches legacy main.py)
pod_norm_all = 2.0 * (pod_coeffs[:, :NUM_AS_MODES] - XX_as_min) / (XX_as_max - XX_as_min) - 1.0
y_active = pod_norm_all @ ss.W1  # (N, 22)

# ac.utils.plotters.sufficient_summary(
#     y_active, qoi_full.reshape(-1, 1),
#     out_label='$C_l$', opts=opts,
#     save_path=os.path.join(RESULTS_DIR, 'sufficient_summary.jpg'),
# )

# POD importance — weighted sum then normalise (matches legacy main.py lines 96-109)
pod_importance = np.sum(ss.eigenvecs[:, :n_active] ** 2 * ss.eigenvals[:n_active, 0], axis=1)
total = pod_importance.sum()
if total > 0:
    pod_importance = pod_importance / total
else:
    pod_importance = np.ones(NUM_AS_MODES) / NUM_AS_MODES

# Determine how many modes are needed to account for 99% of cumulative importance
sorted_idx = np.argsort(pod_importance)[::-1]
cumulative_pct = np.cumsum(pod_importance[sorted_idx]) * 100
idx_99 = np.where(cumulative_pct >= 99.0)[0]
optimal_pod_count_99 = int(idx_99[0]) + 1 if len(idx_99) > 0 else NUM_AS_MODES
print(f'Modes needed for 99 % cumulative importance: {optimal_pod_count_99}')
print(f'Actual cumulative at that point: {cumulative_pct[optimal_pod_count_99 - 1]:.2f} %')

plot_pod_importance(
    NUM_AS_MODES, pod_importance,
    save_dir=os.path.join(RESULTS_DIR, 'Importance'),
    top_n=optimal_pod_count_99,
)

Modes needed for 99 % cumulative importance: 22
Actual cumulative at that point: 99.00 %


/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/utils/visualization.py:178: UserWarning: First parameter to grid() is false, but line properties are supplied. The grid will be enabled.
  ax.grid(False, axis='x', alpha=0.3)


'../results/Case2_NACA4412/Importance/pod_mode_importance_150_top22.jpg'

## 9 · Subspace–Polynomial R² Heatmap

In [12]:
# R² heatmap — matches legacy main.py lines 147-196
n_dim_range  = 22
n_poly_range = 3

# Select top-importance modes (up to optimal_pod_count_99)
n_importance       = np.argsort(pod_importance)[-optimal_pod_count_99:][::-1]
n_dim_range_used   = min(optimal_pod_count_99, n_dim_range)

XX_as_reduced   = pod_norm_all[:, n_importance]        # (N, optimal_pod_count_99)
eigenvecs_reduced = ss.eigenvecs[n_importance, :]      # (optimal_pod_count_99, 150)

print(f'AS dim range: 1–{n_dim_range_used},  poly order range: 1–{n_poly_range}')

heatmap_path, heatmap_data = plot_subspace_polynomial_heatmap(
    XX_as_reduced,
    qoi_full.reshape(-1, 1),
    eigenvecs_reduced,
    n_dim_range=n_dim_range_used,
    n_poly_range=n_poly_range,
    save_dir=os.path.join(RESULTS_DIR, 'Heatmap'),
)

r2_matrix = heatmap_data['r2_matrix']
best_idx       = np.unravel_index(np.nanargmax(r2_matrix), r2_matrix.shape)
best_dim, best_poly = best_idx[0] + 1, best_idx[1] + 1
best_r2        = r2_matrix[best_idx]
print(f'Best R²={best_r2:.6f}  →  subspace dim={best_dim}, poly order={best_poly}')
print('R² matrix:\n', r2_matrix)

AS dim range: 1–22,  poly order range: 1–3


/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  poly_weights = np.linalg.lstsq(B, f)[0]
/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  poly_weights = np.linalg.lstsq(B, f)[0]
/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precisio

Best R²=0.975334  →  subspace dim=19, poly order=3
R² matrix:
 [[0.86709667 0.88307218 0.88379919]
 [0.86862558 0.91162945 0.9209134 ]
 [0.88322646 0.91468557 0.92546306]
 [0.89903237 0.93841079 0.94612975]
 [0.90157903 0.94300968 0.94924719]
 [0.91023944 0.94602744 0.95607358]
 [0.91218241 0.94673454 0.95766065]
 [0.91342693 0.94941894 0.96152064]
 [0.91900581 0.95171111 0.96298905]
 [0.91995448 0.95336072 0.96535594]
 [0.92076481 0.95645932 0.96892405]
 [0.92077473 0.95793429 0.97047479]
 [0.92182685 0.96077873 0.97216144]
 [0.92305468 0.96188897 0.97402237]
 [0.92371068 0.96158758 0.97363696]
 [0.92400159 0.9636072  0.97393044]
 [0.92429935 0.9651621  0.97439202]
 [0.92431059 0.96657153 0.97469116]
 [0.93066455 0.96735058 0.97533427]
 [0.9306155  0.96830922 0.97259343]
 [0.93372523 0.9704676  0.96957157]
 [0.9340278  0.97306025 0.96578223]]


## 10 · Activity Scores and Interaction Heatmap

In [13]:
# Activity scores — matches legacy main.py lines 314-338
top_n = optimal_pod_count_99
alpha_D = (ss.eigenvals[:n_active].reshape(1, n_active) * ss.eigenvecs[:, :n_active] ** 2).sum(axis=1)
alpha_D_top = alpha_D[:top_n]
print(f'Activity scores (first {top_n}): {alpha_D_top}')
print('Normalised:                      ', alpha_D_top / alpha_D_top.sum())

# Lower-triangle modal interaction heatmap — matches legacy main.py lines 343-454
# NACA 4412 uses scientific-notation colorbar and formula without hats
heatmap_fig = plot_interaction_heatmap(
    ss.eigenvecs,
    ss.eigenvals,
    pod_importance,
    n_active=n_active,
    top_n=top_n,
    save_dir=os.path.join(RESULTS_DIR, 'Activity_Score'),
    use_scientific_colorbar=True,
    cbar_formula=r'$\sum_{i=1}^{n} \, \lambda_i \, (\boldsymbol{w}_i \boldsymbol{w}_i^T)$',
)
print(f'Interaction heatmap saved to {heatmap_fig}')

Activity scores (first 22): [1.40177071e+04 7.45649177e+04 1.46695659e+03 4.71100259e+03
 8.94905364e+02 7.22802444e+02 7.25356595e+02 1.58377885e+02
 2.92681095e+02 2.61853193e+02 1.58343132e+01 4.44923691e+01
 1.30601660e+02 1.46018013e+02 2.25410704e+02 3.74198468e+01
 3.53856572e+01 7.25569884e+02 6.14621510e+01 1.52234737e+01
 2.82609219e+02 2.23011818e+01]
Normalised:                       [1.40798147e-01 7.48952890e-01 1.47345617e-02 4.73187541e-02
 8.98870379e-03 7.26004932e-03 7.28570399e-03 1.59079603e-03
 2.93977864e-03 2.63013373e-03 1.59044696e-04 4.46894993e-04
 1.31180310e-03 1.46664968e-03 2.26409421e-03 3.75856412e-04
 3.55424388e-04 7.28784632e-03 6.17344685e-04 1.52909237e-04
 2.83861363e-03 2.23999906e-04]
Interaction heatmap saved to ../results/Case2_NACA4412/Activity_Score/param_interaction_heatmap_lower_top22.jpg


## 11 · Polynomial Response Surface

In [ ]:
from lib.active_subspaces.utils.rs import PolynomialApproximation
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# Project onto best_dim-dimensional reduced subspace (matches legacy main.py lines 217-219)
y_reduced = XX_as_reduced.dot(ss.eigenvecs[n_importance, :best_dim])

X_train, X_test, f_train, f_test = train_test_split(
    y_reduced, qoi_full.reshape(-1, 1), test_size=0.2, random_state=42
)

# Cross-validate polynomial order 1–3 (matches legacy main.py lines 233-256)
n_values, r2_values, rmse_values = [], [], []
best_cv_score, best_cv_rmse, best_cv_n = -1, float('inf'), 1

for n in range(1, 4):
    rs_cv = PolynomialApproximation(N=n)
    rs_cv.train(X_train, f_train)
    pred = rs_cv.predict(X_test)[0]
    score = r2_score(f_test, pred)
    rmse  = np.sqrt(mean_squared_error(f_test, pred))
    print(f'N={n}: R²={score:.6f}, RMSE={rmse:.8f}')
    n_values.append(n); r2_values.append(score); rmse_values.append(rmse)
    if score > best_cv_score or (abs(score - best_cv_score) < 1e-4 and rmse < best_cv_rmse):
        best_cv_score, best_cv_rmse, best_cv_n = score, rmse, n

print(f'\nBest poly order (CV): N={best_cv_n}, R²={best_cv_score:.6f}')

plot_polynomial_cv(
    n_values, r2_values, rmse_values, best_cv_n, best_cv_score, best_cv_rmse,
    save_dir=os.path.join(RESULTS_DIR, 'Polynomial_CV'),
)

# Train final response surface using best_poly from the R² heatmap
RS = PolynomialApproximation(N=best_poly)
RS.train(X_train, f_train)
print(f'Final RS trained with N={best_poly}, train R²={RS.Rsqr:.6f}')

# Build 2-D grid for visualisation (first 2 active dimensions)
x_min, x_max = y_reduced[:, 0].min(), y_reduced[:, 0].max()
y_min, y_max = y_reduced[:, 1].min(), y_reduced[:, 1].max()
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 50),
                     np.linspace(y_min, y_max, 50))
grid_2d = np.vstack([xx.ravel(), yy.ravel()]).T
n_dims = y_reduced.shape[1]
grid_full = np.zeros((grid_2d.shape[0], n_dims))
grid_full[:, :2] = grid_2d
zz = RS.predict(grid_full)[0].reshape(xx.shape)

plot_response_surface_2d(
    xx, yy, zz, y_reduced, qoi_full,
    results_dir=os.path.join(RESULTS_DIR, 'PRS'),
)

validate_response_surface(
    RS, X_test, f_test,
    save_dir=os.path.join(RESULTS_DIR, 'RS_Validation'),
)

compare_rom_fom_predictions(
    y_reduced, qoi_full, RS,
    qoi_label='$C_l$',
    geometry='naca4412',
    total_start_time=600.0,
    plot_start_time=750.0,
    plot_end_time=800.0,
    dt=0.04,
    scatter_xlim=(0.49, 1.1),
    save_dir=os.path.join(RESULTS_DIR, 'ROM_FOM'),
)

N=1: R²=0.930665, RMSE=0.03490480


/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  poly_weights = np.linalg.lstsq(B, f)[0]
/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  poly_weights = np.linalg.lstsq(B, f)[0]


N=2: R²=0.967351, RMSE=0.02395219


/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  poly_weights = np.linalg.lstsq(B, f)[0]


N=3: R²=0.975334, RMSE=0.02081874

Best poly order (CV): N=3, R²=0.975334


/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  poly_weights = np.linalg.lstsq(B, f)[0]


Final RS trained with N=3, train R²=0.991742

Test samples: 1000
  R²   = 0.975334
  RMSE = 0.02081874
  MAE  = 0.01571873
Metrics saved to: ../results/Case2_NACA4412/RS_Validation/validation_metrics.txt
ROM vs FOM: R²=0.9884, RMSE=0.014102, MAE=0.149861
Performance metrics saved to: ../results/Case2_NACA4412/ROM_FOM/performance_metrics.txt


{'r2': 0.9883726217130164,
 'rmse': 0.01410171316642395,
 'mae': 0.14986149968775517}